# Part 3 -- Pricing the fixes

Part 2 named six causes of retrieval failure and pointed each at a fix.
This part prices the fixes: what the cause labels say each one could win,
and what each one actually wins when run. The build order at the end
follows the measured numbers.

Code cells are excerpts from the real scripts; outputs are pasted from the
actual runs. Question examples are paraphrased, never quoted.

## 1. What would each fix win?

A fix can be priced two ways:

- **From the labels.** Count the failures that carry the causes this fix
  targets. That is a ceiling -- the most it could plausibly reach.
- **From a run.** Change the scoring so the fix is in effect, re-rank all
  266 questions, and count what moved: failures fixed, and successes
  broken. Part 2's two counterfactual checks are runs of this kind; the
  keyword run ranks every article by plain word overlap (BM25) instead of
  the embedding.

In [ ]:
for fix in CANDIDATES:
    price[fix] = dict(ceiling=labeled_ceiling(fix), measured=rerun(fix))

fix                       aimed at                        ceiling   measured
keyword ranking (BM25)    the keyword gap                     137   fixes 101
best-sentence scoring     the fact drowns + the lean           96   fixes  60, breaks 19
drop the unlisted copies  near-copies crowd the list           99   fixes  20, breaks  0
rewrite the question      the question leans to one half       67   (section 3)


Two notes on the table. Keyword ranking has no cause of its own, so its
ceiling is every failure except the ones marked "nothing to grab" -- the
questions that give keywords nothing to work with. And its measured
number counts rescues only: it was run as a preview, so the successes it
might break are not counted yet -- the actual build in Part 4 will count
them.

**The labels over-count, on every row.** The near-copy label is on five
times more failures than removing the copies was able to fix.

## 2. The wins overlap

Many failures are rescued by more than one run, so adding the measured
column double-counts. The table reads in build order: what each fix
rescues on its own, how many of those the fixes above it already rescue,
and what it adds new.

In [ ]:
returned = {fix: rescued_set(fix) for fix in RUNS}
show_union(returned)

fix                        rescues on its own   already rescued above   adds new
keyword ranking (BM25)                    101                       -        101
best-sentence scoring                      60                      43        +17
drop the unlisted copies                   20                      15         +5

together (rescued by at least one fix)    123    of the 171 failures


## 3. Can a rewrite fix the lean?

One cause had no run to price: the question leans to one half -- its
single vector sits close to one of the two needed articles while the
other sinks. The obvious fix is cheap: rewrite the question to stress
the buried half. We measured two rewrites, with a control group:

- **re-emphasis** -- restate the question, stressing the buried half,
  using only words the question already has.
- **cheat rewrite** -- also inject a few words taken from the buried
  article itself. A live system has no such words (they are the thing
  being searched for), so this is a deliberate cheat that measures a
  ceiling.
- **control** -- failures WITHOUT the lean, given the same two rewrites.

In [ ]:
for group in (lean_failures, control):
    flips[group] = rescore(rewrites(group))   # a flip = failure becomes success

variant          lean failures (61)          control, no lean (30)
re-emphasis       2% flip,   0 median lift    13% flip,   0 median lift
cheat rewrite    31% flip, +21 median lift    57% flip, +13 median lift

lift = ranks gained by the buried article


Re-emphasis moved almost nothing: the words were already in the question,
and the embedding had already weighed them. Only new information lifted
the buried article. And the control group flips more often (its articles
start closer to the top, so the same lift crosses into the top-10 more
easily) -- the effect is not specific to the lean.

So the cheap rewrite is ruled out, and a real rewrite fix has to split
the question and retrieve each half separately -- decomposition, a
bigger build, deferred.

## 4. Which failures does no fix rescue?

Part 2 answered this from the labels: it counted the failures whose
causes point at no fix on the list. Now every fix has a measured rescue
set, so the question has a direct answer: take all the failures and
remove everything some run rescues.

In [ ]:
no_fix = [q for q in failures if all(q not in returned[f] for f in RUNS)]

Part 2's label count                              28
measured: rescued by no run                       48 of 171

where the difference comes from:
  counted as unfixable, but a run rescues it      20    (18 by keywords)
  counted as fixable, but no run rescues it       40


The label count made two kinds of error:

- **It labeled some failures unfixable, while a run was able to fix
  them.**
- **It labeled some failures fixable, while no run was able to fix
  them.**

Part 2's count was our own label arithmetic; the measured count replaces
it.

In [ ]:
profile(no_fix)

the 48, profiled
  carry "nothing to grab"      20    of that cause's 34 -- keywords cannot reach these
  carry two or more causes     34
  severity                     30 deep / 8 found-nothing / 10 near-miss
  closest listed set           completes at rank 31 (median); worst 163


Here is one of them, with all three runs attempted on it:

In [ ]:
q = no_fix[k]                    # paraphrased in the output
attempt_all(q)

question (paraphrased): how much time passed between a privacy group's
rally about data-handling risks and a property firm's announcement that
one of its surveillance cameras had failed?

needs two events:    the rally (event 2) + the malfunction report (event 5)
closest listed set:  completes at rank 24 -- one malfunction-event copy holds it down
causes it carries:   leans to one half + nothing to grab + near-copies crowd

the three runs, on this question:
  keyword ranking           the copy drops to rank 31 -- its words fit other articles better
  best-sentence scoring     rank 22 -- closer, still outside the top-10
  drop the unlisted copies  16 near-copies removed -- still no listed set completes


A time-span question in a storyline where every article shares the same
surveillance vocabulary: the words the question gives fit dozens of
articles, and the two it needs are held down by their own reworded
copies. Failures like this need decomposition, or a fix we have not
thought of yet.

## 5. The build order

From the numbers above:

1. **Keyword ranking mixed into the scoring, first.** The largest
   measured win, and the cheapest build.
2. **Best-sentence scoring second, as an added signal** next to the
   whole-article score rather than a replacement -- to keep its wins
   without the successes it breaks when it replaces the score outright.
3. **De-duplicate the near-copies third.** Small, and breaks nothing.
4. **Decomposition deferred.** Section 3 priced the cheap version at
   almost nothing; the real version is a different size of work.

(To be continued: the keyword mix built and measured -- how many of the
failures come back.)